## Dictionary-based Classification (kununu)

Dictionary-based classification of German Kununu reviews into Herzberg's 14 two-factor theory categories using a German keyword lexicon.

Results are written to `lex_proscons_kununu` (same schema as `llm_proscons_kununu`).

In [115]:
import sqlite3
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import cohen_kappa_score

DB_PATH = "PJDB.db"

FACTORS_14 = [
    "company_policy_administration", "supervision", "interpersonal_relations",
    "working_conditions", "salary", "personal_life", "job_security",
    "achievement", "recognition", "work_itself", "responsibility",
    "advancement", "possibility_of_growth", "status",
]
LABELS = {
    "company_policy_administration": "Company Policy",
    "supervision":                   "Supervision",
    "interpersonal_relations":       "Interpersonal Relations",
    "working_conditions":            "Working Conditions",
    "salary":                        "Salary",
    "personal_life":                 "Work-Life Balance",
    "job_security":                  "Job Security",
    "achievement":                   "Achievement",
    "recognition":                   "Recognition",
    "work_itself":                   "Work Itself",
    "responsibility":                "Responsibility",
    "advancement":                   "Advancement",
    "possibility_of_growth":         "Growth",
    "status":                        "Status",
}
HYGIENE = {
    "company_policy_administration", "supervision", "interpersonal_relations",
    "working_conditions", "salary", "personal_life", "job_security", "status"
}

with sqlite3.connect(DB_PATH) as conn:
    df = pd.read_sql(
        "SELECT id AS review_id, pros, cons FROM REVIEWS_GER", conn)

print(f"Reviews loaded: {len(df):,}")

Reviews loaded: 19,370


### Lexicon

In [116]:
# Keyword lexicon: diagnostic German terms per factor
LEXICON = {
    "company_policy_administration": [
        "unternehmenspolitik", "firmenpolitik", "personalabteilung", "personalabt",
        "bürokratie", "bürokrat", "hierarchisch", "hierarchie",
        "entscheidungsweg", "unternehmensführung", "betriebsrat",
        "compliance", "regelwerk", "interne kommunikation",
        "kommunikationsstruktur", "interne prozesse", "organisationsstruktur",
        "konzernstruktur", "flache hierarchie", "kurze wege",
        "lange wege", "lange entscheidungswege", "politik im unternehmen",
        "unternehmensstrategie", "hr-prozess", "hr prozesse",
        "managementstruktur", "silodenken",
    ],
    "supervision": [
        "vorgesetzter", "vorgesetzte", "führungsstil", "teamleiter",
        "abteilungsleiter", "direktem vorgesetzten", "direkten vorgesetzten",
        "mein chef", "meine chefin", "meinem chef", "meiner chefin",
        "führungsqualität", "führungskompetenz", "vorgesetzenverhalten",
        "führungskräfte", "schlechte führung", "gute führung",
        "toxische führung", "toxischer manager", "toxische manager",
        "kompetenter vorgesetzter", "inkompetenter vorgesetzter",
    ],
    "interpersonal_relations": [
        "kollegen", "kolleginnen", "kolleg", "teamgeist", "zusammenhalt",
        "arbeitsklima", "betriebsklima", "kollegialität", "wir-gefühl",
        "wir gefühl", "teamatmosphäre", "gemeinschaft", "miteinander",
        "gutes team", "nette kollegen", "tolle kollegen", "super kollegen",
        "atmosphäre", "umgang mit kollegen", "umgang unter kollegen",
        "soziales umfeld", "kollegiale", "teamspirit", "teamzusammenhalt",
    ],
    "working_conditions": [
        "büroausstattung", "büroeinrichtung", "ausstattung", "equipment",
        "hardware", "software", "arbeitsmittel", "arbeitsplatz",
        "homeoffice", "home office", "remote work", "remote arbeiten",
        "überstunden", "workload", "arbeitsbelastung", "arbeitszeiten",
        "arbeitszeit", "zeitdruck", "arbeitsdruck", "stressig",
        "it-ausstattung", "it ausstattung", "laptop", "arbeitsmaterial",
        "räumlichkeiten", "großraumbüro", "open space", "lärmpegel",
        "arbeitspensum", "arbeitslast", "burnout",
    ],
    "salary": [
        "gehalt", "lohn", "vergütung", "bonus", "benefits", "benefit",
        "bezahlung", "urlaubsgeld", "weihnachtsgeld", "tarifvertrag",
        "gehaltserhöhung", "einkommen", "entlohnung", "provision",
        "betriebsrente", "altersvorsorge", "sonderzahlung", "gehälter",
        "löhne", "gehaltsgefüge", "gehaltsstruktur", "lohnerhöhung",
        "gute bezahlung", "schlechte bezahlung", "faire bezahlung",
        "marktgerechtes gehalt", "unterbezahlt", "übertariflich",
    ],
    "personal_life": [
        "work-life-balance", "work life balance", "freizeit", "privatleben",
        "flexibilität", "flexible arbeitszeit", "flexible arbeitszeiten",
        "teilzeit", "elternzeit", "mobiles arbeiten", "heimarbeit",
        "wochenende", "feierabend", "urlaubstage", "familienfreundlich",
        "familienfreundlichkeit", "ausgleich zur arbeit", "gleitzeit",
        "vertrauensarbeitszeit", "freiheiten bei der arbeit",
    ],
    "job_security": [
        "jobsicherheit", "stellensicherheit", "kündigung", "entlassung",
        "stellenabbau", "insolvenz", "unbefristet", "befristet",
        "werksschließung", "umstrukturierung", "restrukturierung",
        "outsourcing", "zukunftsperspektive", "sichere stelle",
        "arbeitsplatzsicherheit", "standortsicherheit", "kündigungsschutz",
        "massenentlassung", "stellenabbau", "keine perspektive",
    ],
    "achievement": [
        "erfolgserlebnis", "erfolgserlebnisse", "eigene erfolge",
        "stolz auf", "herausforderungen meistern", "leistung erbracht",
        "ziel erreicht", "meilenstein", "projekt erfolgreich abgeschlossen",
        "erfüllung", "sinnvolle arbeit", "selbstbestätigung",
        "schöne ergebnisse", "tolle ergebnisse", "man sieht ergebnisse",
        "eigene leistung", "etwas bewegt", "etwas bewirkt",
    ],
    "recognition": [
        "anerkennung", "wertschätzung", "lob", "gelobt", "anerkannt",
        "dankbarkeit", "danke sagen", "respekt", "würdigung",
        "wertgeschätzt", "keine wertschätzung", "nicht wertgeschätzt",
        "ignoriert", "feedbackkultur", "feedback kultur",
        "mitarbeiter werden nicht wertgeschätzt", "nicht anerkannt",
        "leistung wird anerkannt", "leistung wird nicht anerkannt",
    ],
    "work_itself": [
        "interessante aufgaben", "spannende aufgaben", "abwechslungsreiche aufgaben",
        "aufgabengebiet", "aufgabenbereich", "arbeitsinhalt", "tätigkeitsfeld",
        "vielfältige aufgaben", "kreative aufgaben", "sinnvolle aufgaben",
        "herausfordernde aufgaben", "interessante projekte", "spannende projekte",
        "tolle aufgaben", "schöne aufgaben", "abwechslungsreich",
        "spannende themen", "interessantes themenfeld", "bedeutsame arbeit",
        "sinnstiftend",
    ],
    "responsibility": [
        "verantwortung", "autonomie", "eigenverantwortlich",
        "selbstständig arbeiten", "handlungsspielraum", "gestaltungsspielraum",
        "eigenverantwortung", "entscheidungsfreiheit", "eigene entscheidungen",
        "viel freiheit", "eigene projekte", "freiraum", "eigeninitiative",
        "selbstständigkeit", "vertrauen in mitarbeiter",
    ],
    "advancement": [
        "beförderung", "aufstieg", "karrierechancen", "karrieremöglichkeiten",
        "aufstiegsmöglichkeiten", "karrierepfad", "weiterkommen",
        "aufgestiegen", "befördert", "karriereentwicklung",
        "keine aufstiegsmöglichkeiten", "gute karrieremöglichkeiten",
        "karriereperspektiven", "beförderungsmöglichkeit",
    ],
    "possibility_of_growth": [
        "weiterbildung", "fortbildung", "schulung", "schulungen",
        "kurs", "kurse", "seminar", "seminare", "workshop", "workshops",
        "persönliche entwicklung", "lernmöglichkeiten", "weiterentwicklung",
        "qualifizierung", "zertifikat", "zertifikate", "trainingsangebot",
        "weiterlernen", "entwicklungsmöglichkeiten", "bildungsangebote",
        "fachliche weiterentwicklung", "neue fähigkeiten", "neues lernen",
    ],
    "status": [
        "prestige", "image des unternehmens", "unternehmensimage",
        "bekanntheit", "renommee", "ansehen", "dax", "global player",
        "weltkonzern", "markenname", "bekannte firma", "bekanntes unternehmen",
        "namhaftes unternehmen", "renommiertes unternehmen", "namhaft",
        "renommiert", "markenimage", "arbeitgeberimage",
    ],
}

print("Lexicon loaded:")
for f in FACTORS_14:
    print(f"  {LABELS[f]:<26}: {len(LEXICON[f]):>2} terms")

Lexicon loaded:
  Company Policy            : 28 terms
  Supervision               : 22 terms
  Interpersonal Relations   : 24 terms
  Working Conditions        : 31 terms
  Salary                    : 28 terms
  Work-Life Balance         : 20 terms
  Job Security              : 20 terms
  Achievement               : 18 terms
  Recognition               : 19 terms
  Work Itself               : 20 terms
  Responsibility            : 15 terms
  Advancement               : 14 terms
  Growth                    : 23 terms
  Status                    : 18 terms


### Classification

In [ ]:
import spacy

# Load German model; disable unused components for speed
nlp = spacy.load("de_core_news_sm", disable=["parser", "ner"])

def lemmatize_series(series, batch_size=256):
    """Batch-lemmatize a pandas Series; returns list of joined lemma strings"""
    texts = series.fillna("").str[:3000].tolist()
    result = []
    for doc in nlp.pipe(texts, batch_size=batch_size):
        result.append(" ".join(tok.lemma_.lower() for tok in doc if not tok.is_space))
    return result

df["pros_lemma"] = lemmatize_series(df["pros"])
df["cons_lemma"] = lemmatize_series(df["cons"])
print(f"Done. Example:\n  original : {df['pros'].iloc[1]}\n  lemmatized: {df['pros_lemma'].iloc[1]}")

# Lemmatize lexicon keywords so both sides are on the same lemma form
def lemmatize_phrase(phrase):
    return " ".join(tok.lemma_.lower() for tok in nlp(phrase) if not tok.is_space)

LEXICON_LEMMA = {
    f: list(dict.fromkeys(lemmatize_phrase(kw) for kw in kws))  # deduplicate after lemmatization
    for f, kws in LEXICON.items()
}

for f in ["supervision", "interpersonal_relations", "salary", "possibility_of_growth"]:
    print(f"  {LABELS[f]:<26}: {LEXICON_LEMMA[f][:4]}")

In [ ]:
def classify_text(text, keywords):
    """Returns 1 if at least one keyword is found in the lemmatized text, else 0"""
    if not text:
        return 0
    return int(any(kw in text for kw in keywords))

# Classify using lemmatized text + lemmatized keywords
for f in FACTORS_14:
    kws = LEXICON_LEMMA[f]
    df[f"p_{f}"] = df["pros_lemma"].apply(lambda t: classify_text(t, kws))
    df[f"c_{f}"] = df["cons_lemma"].apply(lambda t: classify_text(t, kws))

print(f"Classification complete. N = {len(df):,} reviews.")

# prevalence overview
print(f"\n{'Factor':<26}  {'% Pros':>7}  {'% Cons':>7}")
print("-" * 46)
for f in FACTORS_14:
    pp = df[f"p_{f}"].mean() * 100
    pc = df[f"c_{f}"].mean() * 100
    print(f"{LABELS[f]:<26}  {pp:>7.1f}  {pc:>7.1f}")

### Write to Database

In [ ]:
# Write results to SQLite table
col_defs = ", ".join(
    [f"p_{f} INTEGER DEFAULT 0, c_{f} INTEGER DEFAULT 0" for f in FACTORS_14]
)

with sqlite3.connect(DB_PATH) as conn:
    conn.execute("DROP TABLE IF EXISTS lex_proscons_kununu")
    conn.execute(f"""
        CREATE TABLE lex_proscons_kununu (
            review_id INTEGER PRIMARY KEY,
            {col_defs}
        )
    """)
    factor_cols = [f"{px}_{f}" for f in FACTORS_14 for px in ["p", "c"]]
    insert_cols = ["review_id"] + factor_cols
    placeholders = ", ".join(["?"] * len(insert_cols))
    col_str = ", ".join(insert_cols)
    rows = [
        tuple([int(row["review_id"])] + [int(row[c]) for c in factor_cols])
        for _, row in df.iterrows()
    ]
    conn.executemany(
        f"INSERT INTO lex_proscons_kununu ({col_str}) VALUES ({placeholders})", rows
    )

print(f"Table lex_proscons_kununu written: {len(df):,} rows, {2*len(FACTORS_14)} factor columns.")

Table lex_proscons_kununu written: 19,370 rows, 28 factor columns.
